# Regularization Cheat Sheet for Economics/Econometrics Applications (scikit-learn)

Quick reference for regularized regression on economic data: wage equations, growth regressions, demand estimation, or any cross-sectional/panel dataset with many candidate controls. Run the snippet cells and adapt to your own project.

## 1. Why economics leans on regularization

- **Many candidate controls, limited observations.** Cross-country growth regressions may have 40+ candidate explanatory variables and fewer than 100 countries — the classic setting Sala-i-Martin (1997) called "I just ran two million regressions," motivating principled, automated variable selection instead of ad hoc specification search.
- **Multicollinearity is structural, not incidental.** Age, education, and experience are mechanically related by construction (`age ≈ education + experience + years before working`); GDP, investment, and capital stock move together over time. Ridge is historically motivated by exactly this problem in econometrics.
- **Log-linear models are the norm.** Wage, price, and output regressions typically use `log(y)` as the target so that OLS/Ridge/Lasso coefficients are approximately interpretable as percentage effects.

## 2. Regression regularization vs. classification regularization

Economic outcomes (wages, GDP growth, prices, quantities) are usually **continuous**, so you'll reach for `Ridge`/`Lasso`/`ElasticNet` (or their `*CV` variants) far more often than `LogisticRegression`. Classification does show up (loan default, labor-force participation, poverty status) — see the general-purpose cheat sheet for that side.

| | `Ridge` / `Lasso` / `ElasticNet` | `LogisticRegression` |
|---|---|---|
| Hyperparameter | `alpha` (bigger = stronger penalty) | `C` (bigger = weaker penalty) |
| Typical economics use | Wage/growth/demand regressions | Default, labor-force participation, poverty classification |
| Metric to tune on | `neg_mean_squared_error`, `r2` | `roc_auc`, `f1`, `accuracy` |

## 3. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV, ElasticNetCV,
)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
print("Imports OK")

## 4. Standard pipeline skeleton

In [ ]:
def scale_and_split(X_raw, y, test_size=0.2, random_state=42):
    scaler = StandardScaler().fit(X_raw)
    X = scaler.transform(X_raw)
    return (*train_test_split(X, y, test_size=test_size, random_state=random_state), scaler)

print("scale_and_split defined")

## 5. Fitting each estimator

In [ ]:
# Manual alpha
ridge = Ridge(alpha=1.0)          # ridge.fit(X_train, y_train)
lasso = Lasso(alpha=0.1)          # lasso.fit(X_train, y_train)
enet  = ElasticNet(alpha=0.1, l1_ratio=0.5)

# Auto-tuned (searches alpha via internal CV — usually the better default choice)
ridge_cv = RidgeCV(alphas=np.logspace(-3, 4, 100), cv=5)
lasso_cv = LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, max_iter=10000)
enet_cv  = ElasticNetCV(alphas=np.logspace(-4, 1, 50), l1_ratio=[.1,.3,.5,.7,.9,.95,.99], cv=5, max_iter=10000)

print("Reference only — call .fit(X_train, y_train)")

## 6. Reading results

| Attribute | Meaning |
|---|---|
| `.coef_` | standardized coefficients if X was scaled — convert back to "per unit" terms with `coef_ / scaler.scale_` if you need raw-unit interpretation |
| `.alpha_` (CV estimators) | the selected regularization strength |
| `.intercept_` | never penalized; not comparable across scaled/unscaled fits directly |
| `r2_score` | fraction of variance explained — economics convention, always report alongside MSE |
| `mean_absolute_error` | easier to communicate to non-technical audiences than MSE (same units as `y`) |

## 7. Log-linear interpretation cheat sheet

If your target is `log(y)` (standard for wages, prices, GDP):

- A coefficient `β` on a continuous regressor `x` means: a one-unit increase in `x` is associated with an approximate `100·β`% change in `y` (exact: `100·(e^β − 1)%`).
- A coefficient `β` on a 0/1 dummy means: being in that group is associated with an approximate `100·β`% difference in `y` relative to the reference group.
- These interpretations assume **unstandardized** coefficients. If you scaled `X` for regularization (as you should), report tuned coefficients from a matching **unstandardized** OLS/Ridge/Lasso fit for the write-up, or manually rescale: `raw_coef = standardized_coef / scaler.scale_[j]`.

## 8. Variable-selection workflow for control variables (double/debiased Lasso, simplified)

A common applied-economics pattern when you care about **one specific coefficient** (e.g., a treatment or policy variable) but have many nuisance controls:

1. Run `LassoCV` of the outcome on all controls (excluding the variable of interest) → note which controls survive.
2. Run `LassoCV` of the variable of interest on all controls → note which controls survive.
3. Take the **union** of the two survived-control sets.
4. Run plain OLS of the outcome on the variable of interest **plus** that union of controls.
5. Report the OLS coefficient on the variable of interest from step 4 — not a Lasso coefficient — because OLS at this stage is (approximately) unbiased, while a direct Lasso coefficient would be shrunk toward zero.

```python
from sklearn.linear_model import LassoCV, LinearRegression

# y: outcome, d: treatment/policy variable of interest, W: matrix of controls
sel_y = LassoCV(cv=5).fit(W, y).coef_ != 0
sel_d = LassoCV(cv=5).fit(W, d).coef_ != 0
selected = sel_y | sel_d

X_final = np.column_stack([d, W[:, selected]])
final_model = LinearRegression().fit(X_final, y)
treatment_effect = final_model.coef_[0]   # approximately debiased
```

## 9. Common pitfalls specific to economic data

- **Reporting a Lasso/Ridge coefficient as "the causal effect."** These estimators are biased toward zero on purpose; use them for prediction or (via double-selection) control selection, not as your final causal estimate.
- **Forgetting mechanical collinearity.** `age`, `education`, `experience` (and `experience²`) will always be correlated by construction in labor data — don't be surprised when regularization redistributes credit among them.
- **Not logging skewed outcomes.** Wages, GDP, and firm size are usually right-skewed; fitting a linear model on the raw level instead of the log can produce a poor fit and coefficients that are much harder to interpret.
- **Standardizing dummy variables (0/1 controls) without noting it.** It's fine to include dummies in `StandardScaler`, but remember a coefficient on a scaled dummy no longer means "the effect of going from 0 to 1" directly — rescale before writing it up.
- **Comparing models across log vs. level targets by MSE.** MSE on `log(y)` and MSE on `y` are not comparable numbers — always state which scale a reported MSE/R² is on.